# 02c. Random Forest versus Isolation Forest

Цель этого notebook — проверить, улучшает ли supervised `RandomForestClassifier` разделение строго размеченных known-neutral и confirmed-pathogenic вариантов по сравнению с текущим one-class `IsolationForest`. Это benchmark, а не автоматическая замена neutral-domain модели в downstream-анализе.

## Дизайн сравнения

- отрицательный класс: Dataset 8 neutral без пересечения с Dataset 9;
- положительный класс: Dataset 9 pathogenic без пересечения с Dataset 8;
- четыре конфликтующие записи исключены;
- 5-fold cross-validation повторена 5 раз; группы разбиения — позиции mtDNA, поэтому разные SNV одной позиции не попадают одновременно в train и test;
- Isolation Forest обучается только на neutral-части train, Random Forest — на обоих классах;
- порог обоих методов определяется как 95-й перцентиль neutral scores на отдельной внутренней calibration-части;
- доверительные интервалы получены bootstrap-перевыборкой позиций, а не отдельных строк.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

ROOT = Path('..').resolve()
RESULT_DIR = ROOT / 'results/model_random_forest_benchmark'
FIGURE_DIR = ROOT / 'results/figures/model_random_forest_benchmark'
BENCHMARK_SCRIPT = ROOT / 'scripts/random_forest_benchmark.py'

# Полный расчёт занимает несколько минут. Поставьте True только для пересчёта.
RUN_BENCHMARK = False
if RUN_BENCHMARK:
    subprocess.run([sys.executable, str(BENCHMARK_SCRIPT)], check=True)

In [ ]:
cohort_audit = pd.read_csv(RESULT_DIR / 'benchmark_cohort_audit.tsv', sep='\t')
metric_summary = pd.read_csv(RESULT_DIR / 'model_metric_summary.tsv', sep='\t')
metric_difference = pd.read_csv(
    RESULT_DIR / 'random_forest_minus_isolation_forest.tsv', sep='\t'
)
feature_importance = pd.read_csv(
    RESULT_DIR / 'random_forest_feature_importance.tsv', sep='\t'
)
display(cohort_audit)

## Основной результат на текущих девяти признаках

На полной панели Random Forest лучше: ROC AUC `0.996` против `0.973`, average precision `0.797` против `0.494`, sensitivity при T95-подобной калибровке `0.988` против `0.857`. Позиционный bootstrap подтверждает положительную разность и для ROC AUC (`+0.024`, 95% CI `0.012–0.039`), и для average precision (`+0.302`, 95% CI `0.193–0.404`).

Из-за распространённости pathogenic-класса около 1% precision остаётся умеренной даже у RF: `0.179` против `0.150`; поэтому обычную accuracy здесь не следует использовать.

In [ ]:
primary_metrics = [
    'roc_auc', 'average_precision', 'balanced_accuracy',
    'sensitivity', 'specificity', 'precision', 'f1', 'mcc',
]
display(
    metric_summary[
        metric_summary['feature_panel'].eq('current_all_9')
        & metric_summary['metric'].isin(primary_metrics)
    ].sort_values(['metric', 'model'])
)

![ROC и precision-recall](../results/figures/model_random_forest_benchmark/roc_pr_comparison.png)

![Сравнение метрик](../results/figures/model_random_forest_benchmark/metric_comparison.png)

## Проверка устойчивости к определению neutral-набора

Dataset 8 включает варианты, отобранные по `lowest_decile_phyloP` и haplogroup-критериям. При этом `phyloP100way` и population rarity входят в исходные девять признаков. Поэтому высокая supervised-метрика может частично означать восстановление правила разметки.

После удаления `phyloP100way` преимущество RF становится небольшим и статистически неустойчивым: ROC AUC `0.961` против `0.947`, average precision `0.285` против `0.265`; оба 95% CI разности включают ноль. На `mlc_score + codon position` Isolation Forest выше по ROC AUC (`0.900` против `0.863`), а на одном `mlc_score` — `0.898` против `0.772`.

In [ ]:
sensitivity_readout = metric_summary[
    metric_summary['metric'].isin(['roc_auc', 'average_precision'])
].pivot_table(
    index=['feature_panel', 'metric'], columns='model', values='value'
).reset_index()
sensitivity_readout['RF_minus_IF'] = (
    sensitivity_readout['random_forest']
    - sensitivity_readout['isolation_forest']
)
display(sensitivity_readout)
display(
    metric_difference[
        metric_difference['metric'].isin(['roc_auc', 'average_precision'])
    ][['feature_panel', 'metric', 'value', 'ci_lower', 'ci_upper']]
)

## Вывод

**Да, Random Forest лучше классифицирует текущие curated neutral/pathogenic labels на полном наборе признаков. Но пока нет оснований заменять им Isolation Forest в основном pipeline.**

Причины: RF решает другую, supervised-задачу; pathogenic train содержит только 84 варианта; neutral labels связаны с частью входных признаков; метрики не подтверждены на независимом внешнем наборе. Практический следующий шаг — внешняя валидация или формирование label set, не определённого через `phyloP`/population frequency. До этого RF лучше хранить как альтернативный benchmark, а не использовать для пересчёта групп и мутационных спектров.